In [1]:
# !pip install QuantumRingsLib

## Imports

In [2]:
from QuantumRingsLib import QuantumRingsProvider
from quantumrings.toolkit.qiskit import QrBackendV2

from qiskit import QuantumCircuit
from qiskit import transpile

import matplotlib.pyplot as plt
import numpy as np
import random
import math

from semiprimes import semiprimes

In [3]:
#testing the file access for the semiprimes is working
for bit_lengths, semiprime_numbers in semiprimes.items():
    print(f"Bit Length: {bit_lengths}, Number: {semiprime_numbers}")

Bit Length: 8, Number: 143
Bit Length: 10, Number: 899
Bit Length: 12, Number: 3127
Bit Length: 14, Number: 11009
Bit Length: 16, Number: 47053
Bit Length: 18, Number: 167659


## Shor Implementation

### Quantum Provider

In [ ]:
qr_provider = QuantumRingsProvider(token ='[redacted]', name='[redacted]')

### Modular Exponentiation

In [5]:
def mod_exp(a, N, qubits):
    # This function implements modular exponentiation as a quantum operation.
    # The function takes four parameters:
    # - `a`: The base of the modular exponentiation.
    # - `N`: The modulus for the operation.
    # - `qubits`: The number of qubits used in the quantum circuit.
    # The function creates a quantum circuit that applies Hadamard gates to all qubits to create superposition,
    # calculates the modular exponentiation for each qubit, and applies phase shifts based on the result.
    # It then converts the circuit into a controlled gate for use in quantum algorithms.
    
    U = QuantumCircuit(qubits)
    for i in range(qubits):
        U.h(i) 
    
    for j in range(qubits):
        a_mod_exp = pow(a, 2**j, N) 
        phase_value = 2 * math.pi * a_mod_exp / N 
        for q in range(qubits):
            U.p(phase_value, q)
    
    U = U.to_gate()
    U.name = f"{a}^x mod {N}"
    c_U = U.control()

    return c_U

### Inverse QFT

In [6]:
def inverse_qft(n_qubits):
    # This function implements the inverse Quantum Fourier Transform (QFT^-1).
    # The inverse QFT is the reverse of the QFT, which is a key component in many quantum algorithms.
    # The function creates a quantum circuit with `n` qubits and performs the following steps:
    # 1. Swaps the qubits to reverse their order, as QFT^-1 requires reversing the qubit order.
    # 2. Applies controlled phase gates (cp) to introduce phase shifts between qubits.
    # 3. Applies Hadamard gates (h) to each qubit to complete the inverse QFT transformation.
    
    qc = QuantumCircuit(n_qubits)

    for qubit in range(n_qubits//2):
        qc.swap(qubit, n_qubits-qubit-1)

    for i in range(n_qubits):
        for j in range(i):
            qc.cp(-math.pi/float(2**(i-j)), j, i)
        qc.h(i)

    qc.name = "QFT^-1"
    
    return qc

def approximate_inverse_qft(n_qubits, precision=3):
    qc = QuantumCircuit(n_qubits)
    # Phase truncation
    for i in range(n_qubits):
        for j in range(max(0, i - precision + 1), i):
            qc.cp(-math.pi/(2**(i-j)), j, i)
        qc.h(i)
    qc.name = "Approximate QFT^-1"    
    return qc

### Main Runtime

Note, all cells above still need to be run for the functions to work correctly in a jupyter notebook

In [7]:
successful_factors = []
a_attempts_count = []
gate_count_list = []

for bit_lengths, semiprime_numbers in semiprimes.items():
    N = semiprime_numbers
    a_attempts = []
    while 1:
        p, q = 1, 1
        
        a = random.randint(2, N - 2)
        if math.gcd(a, N) != 1:
            p = math.gcd(a, N)
            q = N // p

        print("factoring", N, "with a =", a)
        if math.gcd(a, N) > 1:
            p = math.gcd(a, N)
            q = N // p
            break

        n_qubits = (N.bit_length() // 2)
        # n = math.ceil(math.log2(N))
        # source_qubits = n + 2  # Reduce from 2n to n+2 — works reasonably well
        # target_qubits = n      # Minimum needed to mod exp N

        # n_qubits = source_qubits + target_qubits
        # print("n_qubits", n_qubits)

        #initialization
        qc = QuantumCircuit(2 * n_qubits, n_qubits)
        
        for q in range(n_qubits):
            qc.h(q) 
        
        qc.x(n_qubits * 2 - 1)  #the last gate is switched to 1 


        #modular exponentiation
        for q in range(n_qubits):
            qc.append(mod_exp(a, N, n_qubits), [q] + [i + n_qubits for i in range(n_qubits)])


        #inverse QFT
        qc.append(inverse_qft(n_qubits), range(n_qubits))  
        qc.measure(range(n_qubits), range(n_qubits))  

        #backend
        backend = QrBackendV2(qr_provider, num_qubits=qc.num_qubits)
        qc_transpiled = transpile(qc, backend, initial_layout=[i for i in range(qc.num_qubits)])
        job = backend.run(qc_transpiled, shots=1000)
        result = job.result()

        counts = result.get_counts()

        gate_count = qc.size()
        gate_count_list.append(gate_count)

        #classical component
        r = int(max(counts, key=counts.get), 2)
        
        f1 = math.gcd(a**(r//2) - 1, N)
        f2 = math.gcd(a**(r//2) + 1, N)

        #check if the factors are valid
        if f1 == 1 and f2 > 1:
            p = f2
            q = N // f2
        elif f2 == 1 and f1 > 1:
            p = f1
            q = N // f1

        if p != 1 and q != 1 and p * q == N:
            break #found the factors

        a_attempts.append(a) #ignoring repeats so the length is not perfect if the random generator is bad 
        # print(a_attempts)
    gate_count_list.append(gate_count)

    # ----------------------------------------------------------------
    
    print("Successfully factored", N, "into", p, "and", q, "using", gate_count * len(a_attempts), "gates")
    successful_factors.append(N)
    a_attempts_count.append(len(a_attempts))
    print()
    print()
    print()

factoring 143 with a = 73
factoring 143 with a = 28
factoring 143 with a = 75
factoring 143 with a = 87
Successfully factored 143 into 11 and 13 using 42 gates



factoring 899 with a = 383
factoring 899 with a = 122
factoring 899 with a = 405
factoring 899 with a = 345
factoring 899 with a = 420
factoring 899 with a = 135
factoring 899 with a = 700
factoring 899 with a = 109
factoring 899 with a = 161
factoring 899 with a = 395
factoring 899 with a = 344
factoring 899 with a = 136
factoring 899 with a = 88
factoring 899 with a = 116
Successfully factored 899 into 29 and 31 using 221 gates



factoring 3127 with a = 114
factoring 3127 with a = 2961
factoring 3127 with a = 363
factoring 3127 with a = 1538
factoring 3127 with a = 31
factoring 3127 with a = 2741
factoring 3127 with a = 2151
factoring 3127 with a = 3011
factoring 3127 with a = 1806
factoring 3127 with a = 1787
factoring 3127 with a = 224
factoring 3127 with a = 1520
factoring 3127 with a = 1337
factoring 3127 with a = 976


In [8]:
for i, N in enumerate(successful_factors):
    print("Successfully factored", N, "in", a_attempts_count[i], "quantum attempts, for a total of", gate_count_list[i] * a_attempts_count[i], "quantum gates")

Successfully factored 143 in 3 quantum attempts, for a total of 42 quantum gates
Successfully factored 899 in 13 quantum attempts, for a total of 182 quantum gates
Successfully factored 3127 in 33 quantum attempts, for a total of 462 quantum gates
Successfully factored 11009 in 43 quantum attempts, for a total of 602 quantum gates
Successfully factored 47053 in 252 quantum attempts, for a total of 3528 quantum gates
Successfully factored 167659 in 205 quantum attempts, for a total of 3485 quantum gates


-------------